# APIM 경유 Azure OpenAI 호출 테스트

이 노트북은 **APIM Subscription Key** 방식으로 Azure OpenAI를 호출하는 실습/검증용입니다.

- 학생 계정(Entra ID) 불필요
- 필요한 것: APIM Endpoint + Subscription Key
- 관련 문서: [apim-architecture-guide.md](./apim-architecture-guide.md), [apim-setup-guide.md](./apim-setup-guide.md)

## 체크리스트
1. `.env` 파일에 아래 4개 환경변수가 설정되어 있는지 확인
   - `AZURE_OPENAI_ENDPOINT`
   - `AZURE_OPENAI_API_VERSION`
   - `APIM_SUBSCRIPTION_KEY`
   - `AZURE_OPENAI_DEPLOYMENT`
2. 네트워크에서 APIM 도메인(`*.azure-api.net`) 접근 가능한지 확인


## 0. 환경 준비

In [1]:
# 필요한 패키지는 pyproject.toml에 이미 포함되어 있습니다.
# !pip install -q openai python-dotenv requests

import os
from dotenv import load_dotenv

load_dotenv(override=True)  # .env 파일에서 환경변수 로드, 이미 설정된 변수는 덮어쓰기

ENDPOINT      = os.getenv("AZURE_OPENAI_ENDPOINT", "https://{apim name}.azure-api.net")
API_VERSION   = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
SUBSCRIPTION  = os.getenv("APIM_SUBSCRIPTION_KEY", "")
DEPLOYMENT    = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1-mini")
EMBED_DEPLOY  = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")

assert SUBSCRIPTION, "APIM_SUBSCRIPTION_KEY 환경변수가 비어 있습니다. .env 파일을 확인하세요."

print("Endpoint    :", ENDPOINT)
print("API Version :", API_VERSION)
print("Deployment  :", DEPLOYMENT)
print("Sub Key     :", SUBSCRIPTION[:6] + "..." + SUBSCRIPTION[-4:])


Endpoint    : https://{apim name}.azure-api.net
API Version : 2024-12-01-preview
Deployment  : gpt-4.1-mini
Sub Key     : 56af3d...8c7b


## 1. 연결 헬스체크 (raw HTTP)

SDK 없이 `requests`로 APIM 게이트웨이에 직접 호출해 네트워크·키·배포명을 한 번에 확인합니다.


In [2]:
import requests, json

url = f"{ENDPOINT}/openai/deployments/{DEPLOYMENT}/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": SUBSCRIPTION,
}
params = {"api-version": API_VERSION}
payload = {
    "messages": [{"role": "user", "content": "ping"}],
    "max_tokens": 5,
}

r = requests.post(url, headers=headers, params=params, json=payload, timeout=30)
print("HTTP", r.status_code)
try:
    print(json.dumps(r.json(), ensure_ascii=False, indent=2)[:800])
except Exception:
    print(r.text[:800])

assert r.status_code == 200, "헬스체크 실패: 상단 체크리스트 및 apim-setup-guide.md Step 8을 확인하세요."


HTTP 200
{
  "choices": [
    {
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "protected_material_code": {
          "detected": false,
          "filtered": false
        },
        "protected_material_text": {
          "detected": false,
          "filtered": false
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
      },
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "message": {
        "annotations": [],
        "content": "pong! How can I",
       


## 2. OpenAI SDK로 Chat Completions

APIM은 `api-key` 헤더 대신 `Ocp-Apim-Subscription-Key`를 요구합니다.
`default_headers`로 주입합니다.


In [3]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_version=API_VERSION,
    api_key="unused",  # APIM 정책이 api-key 헤더를 제거하므로 더미값 OK
    default_headers={"Ocp-Apim-Subscription-Key": SUBSCRIPTION},
)

resp = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[
        {"role": "system", "content": "너는 친절한 한국어 AI 튜터야."},
        {"role": "user",   "content": "대한민국의 수도를 한 문장으로 알려줘."},
    ],
    max_tokens=80,
    temperature=0.3,
)

print(resp.choices[0].message.content)
print("---")
print("usage:", resp.usage)


대한민국의 수도는 서울입니다.
---
usage: CompletionUsage(completion_tokens=9, prompt_tokens=36, total_tokens=45, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


## 3. 스트리밍 응답

In [5]:
stream = client.chat.completions.create(
    model=DEPLOYMENT,
    messages=[{"role": "user", "content": "파이썬으로 피보나치 수열을 1줄로 표현해줘."}],
    max_tokens=120,
    stream=True,
)

for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


```python
fib = lambda n: n if n <= 1 else fib(n-1) + fib(n-2)
```
이 한 줄로 피보나치 수열의 n번째 값을 구할 수 있습니다.  
예: `fib(5)`는 5번째 피보나치 수인 5를 반환합니다.


## 4. 임베딩(Embeddings) 테스트

In [6]:
try:
    emb = client.embeddings.create(
        model=EMBED_DEPLOY,
        input=["APIM은 게이트웨이입니다.", "Managed Identity로 인증을 대행합니다."],
    )
    print("벡터 개수:", len(emb.data))
    print("벡터 차원:", len(emb.data[0].embedding))
    print("usage   :", emb.usage)
except Exception as e:
    print("임베딩 배포가 없거나 허용되지 않았습니다:", e)


벡터 개수: 2
벡터 차원: 3072
usage   : Usage(prompt_tokens=23, total_tokens=23)


## 5. 보안/정책 검증

APIM 정책이 기대대로 동작하는지 확인합니다.


### 5-1. 잘못된 Subscription Key → 401

In [7]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json", "Ocp-Apim-Subscription-Key": "invalid-key-xxx"},
    params=params,
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
print(r.text[:300])
assert r.status_code in (401, 403), "APIM이 잘못된 키를 거부하지 못했습니다."
print("✅ 잘못된 키 거부 확인")


HTTP 401
{ "statusCode": 401, "message": "Access denied due to invalid subscription key. Make sure to provide a valid key for an active subscription." }
✅ 잘못된 키 거부 확인


### 5-2. Subscription Key 누락 → 401

In [8]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json"},
    params=params,
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
print(r.text[:300])
assert r.status_code in (401, 403), "APIM이 키 없는 요청을 거부하지 못했습니다."
print("✅ 키 누락 거부 확인")


HTTP 401
{ "statusCode": 401, "message": "Access denied due to missing subscription key. Make sure to include subscription key when making requests to an API." }
✅ 키 누락 거부 확인


### 5-3. `api-key` 헤더만 보내도 무시되는지

학생이 실수로/의도적으로 원본 `api-key`를 넣어도 APIM 정책이 제거합니다.
Subscription Key 없이는 통과하면 안 됩니다.


In [9]:
r = requests.post(
    url,
    headers={"Content-Type": "application/json", "api-key": "whatever"},
    params=params,
    json=payload,
    timeout=30,
)
print("HTTP", r.status_code)
assert r.status_code in (401, 403), "api-key만으로 통과됨 → 정책 점검 필요"
print("✅ api-key 단독 호출 거부 확인")


HTTP 401
✅ api-key 단독 호출 거부 확인


## 6. 간단 벤치마크 (선택)

평균 지연시간과 실패율을 빠르게 확인합니다. 남용 방지를 위해 10회로 제한합니다.


In [10]:
import time, statistics

N = 10
latencies = []
errors = 0

for i in range(N):
    t0 = time.perf_counter()
    try:
        client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": f"숫자 {i}를 한국어로 써줘."}],
            max_tokens=10,
        )
        latencies.append(time.perf_counter() - t0)
    except Exception as e:
        errors += 1
        print(f"[{i}] error:", e)

if latencies:
    print(f"성공 {len(latencies)}/{N}")
    print(f"평균 지연: {statistics.mean(latencies):.2f}s")
    print(f"최소/최대: {min(latencies):.2f}s / {max(latencies):.2f}s")
print(f"실패: {errors}")


성공 10/10
평균 지연: 1.22s
최소/최대: 0.78s / 2.48s
실패: 0


## 7. 트러블슈팅

| 증상 | 원인 | 해결 |
|------|------|------|
| 401 Access denied due to invalid subscription key | Subscription Key 오타/비활성 | 운영자에게 키 재발급 요청 |
| 404 Resource not found | `deployment-id` 오타 또는 허용 목록 외 모델 | `AZURE_OPENAI_DEPLOYMENT` 확인 |
| 403 Forbidden (IP) | 교육장 외부에서 호출 | IP allowlist 정책 확인 |
| 429 Too Many Requests | 분당/일간 쿼터 초과 | 잠시 후 재시도 또는 운영자 문의 |
| 500/502 | 백엔드/MI 토큰 문제 | `apim-setup-guide.md` Step 3, 6 재점검 |
| SDK: api_key required | SDK가 빈 키를 거부 | `api_key="unused"` 같은 더미 문자열 지정 |
